# Goal Selection in Humans and Machines

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

import helpers as h
from settings import data_dir, out_dir
og_data_dir = data_dir

## Gather data

In [ ]:
plots_dir = "results/figures/"
plotters = {}
filled_markers = Line2D.filled_markers[::-1]

# ============================================================================
# USER CONFIG: edit these to point at your own processed data snapshot.
# `version_dir` selects which experiment variant to analyze (must match a
# subfolder under `out_dir`, e.g. `out_dir/V2/<family>/<model>/learn_df.csv`).
# The branches below are examples from our own experiments (see README.md);
# add/remove branches, model lists, and labels to match your own data.
# ============================================================================
version_dir = ""  # "" # "V2/" # "V3/" # "V4/" # "V5/" # "V6/" # "V7/" # "V8/"
temp = 1.0
cot_nt = True

# A couple of our variants were collected under differently-named raw data
# directories (see README.md); adapt this to your own naming convention, or
# remove it entirely if you only ever analyze one raw data directory.
if version_dir == "V3/":
    data_dir = data_dir.replace("Alchemist", "Acrobat")
    version_dir_for_data_dir = ""
elif version_dir == "V4/":
    data_dir = data_dir.replace("Alchemist", "NonWords")
    version_dir_for_data_dir = ""
else:
    data_dir = og_data_dir
    version_dir_for_data_dir = version_dir

if version_dir == "":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = ["gemini-2.5-pro_cot", "gpt-5_cot", "qwen3-32b_cot"]
    version_suffix  = ""
elif version_dir == "V2/":
    base_models = ["gemini-2.5-pro", "gpt-5", "qwen3-32b"]
    cot_models =  []
    version_suffix = "_persona"
elif version_dir == "V3/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_acrobat"
elif version_dir == "V4/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_nonwords"
elif version_dir == "V5/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_cw48"
elif version_dir == "V6/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_cw120"
elif version_dir == "V7/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_cw24"
elif version_dir == "V8/":
    base_models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []
    version_suffix = "_cw12"

if temp == 0.0:
    base_models = ["gemini-2.5-pro", "claude-sonnet-4.5", "qwen3-32b", "centaur"]
    cot_models = []

if cot_nt:
    cot_models = [model.replace("_cot", "_cot_nt") for model in cot_models]


In [ ]:
model_colors_dict = {
    "gpt-5": "#12a37f", 
    "gpt-5_cot": "#12a37f",
    "gpt-5_cot_nt": "#12a37f",
    "gemini-2.5-pro": "#F4B400", 
    "gemini-2.5-pro_cot": "#F4B400", 
    "gemini-2.5-pro_cot_nt": "#F4B400", 
    "centaur": "#691a5e", 
    "claude-sonnet-4.5": "#da7756",
    "qwen3-32b": "#5a52d9",
    "qwen3-32b_cot": "#5a52d9",
    "qwen3-32b_cot_nt": "#5a52d9",
}

model_markers_dict = {
    "human": "X",
    "gpt-5": "*", 
    "gpt-5_cot": "*",
    "gpt-5_cot_nt": "*",
    "gemini-2.5-pro": "D", 
    "gemini-2.5-pro_cot": "D", 
    "gemini-2.5-pro_cot_nt": "D", 
    "centaur": "h", 
    "claude-sonnet-4.5": "^",
    "qwen3-32b": "P",
    "qwen3-32b_cot": "P",
    "qwen3-32b_cot_nt": "P",
}

model_names_dict = {
    "human": "Humans",
    "gpt-5": "GPT-5", 
    "gpt-5_cot": "GPT-5 CoT",
    "gpt-5_cot_nt": "GPT-5 CoT",
    "gemini-2.5-pro": "Gemini 2.5 Pro", 
    "gemini-2.5-pro_cot": "Gemini 2.5 Pro CoT", 
    "gemini-2.5-pro_cot_nt": "Gemini 2.5 Pro CoT", 
    "centaur": "Centaur", 
    "claude-sonnet-4.5": "Claude Sonnet 4.5",
    "qwen3-32b": "Qwen3 32B",
    "qwen3-32b_cot": "Qwen3 32B CoT",
    "qwen3-32b_cot_nt": "Qwen3 32B CoT",
}

# Available markers: https://matplotlib.org/stable/gallery/lines_bars_and_markers/marker_reference.html

In [ ]:
highlight_examples = True  # Toggle to highlight example participants on sorted metric plots

if temp == 1.0 and version_dir == "":
    example_participants = {
        "human": [28, 173],
        "gemini-2.5-pro": [48, 29],
        "gpt-5": [13, 0],
        "claude-sonnet-4.5": [25, 22],
        "centaur": [58, 84],
        "qwen3-32b": [3, 28],
    }
else:
    example_participants = {}

In [ ]:
human_out_dir = f"{out_dir}/human/"

human_task_df = pd.read_csv(f"{human_out_dir}task_df.csv")
human_train_df = pd.read_csv(f"{human_out_dir}train_df.csv")
human_learn_df = pd.read_csv(f"{human_out_dir}learn_df.csv")
human_test_df = pd.read_csv(f"{human_out_dir}test_df.csv")

human_plotter = h.Plotter(
    task_df=human_task_df,
    train_df=human_train_df,
    learn_df=human_learn_df,
    test_df=human_test_df,
)
human_plotter.main_color = "k"
human_plotter.marker = filled_markers[0]
human_plotter.get_metrics()

plotters["human"] = human_plotter

## Plots

In [ ]:
for model in base_models + cot_models:
    family = h.get_family(model)

    task_df = pd.read_csv(f"{out_dir+version_dir}/{family}/{model}/task_df.csv")
    train_df = pd.read_csv(f"{out_dir+version_dir}/{family}/{model}/train_df.csv")
    learn_df = pd.read_csv(f"{out_dir+version_dir}/{family}/{model}/learn_df.csv")
    test_df = pd.read_csv(f"{out_dir+version_dir}/{family}/{model}/test_df.csv")
    temp_vals = task_df.temperature.unique()
    if len(temp_vals) <= 1:
        print(f"{model} only has one temperature setting: {temp_vals[0]}.")
        temp_plotter = h.Plotter(task_df, train_df, learn_df, test_df)
        temp_plotter.main_color = model_colors_dict[model]
        temp_plotter.marker = model_markers_dict[model]
        temp_plotter.get_metrics()
        plotters[f"{model}"] = temp_plotter
    else:
        print(f"{model} has multiple temperature settings: {temp_vals}.")
        print(f"    Only processing temperature={temp}.")
        temp_plotter = h.Plotter(task_df.query("temperature==@temp"),
                                train_df.query("phase==0 & temperature==@temp"),
                                learn_df.query("phase==1 & temperature==@temp"),
                                test_df.query("phase==2 & temperature==@temp"))

        temp_plotter.main_color = model_colors_dict[model]
        temp_plotter.marker = model_markers_dict[model]
        temp_plotter.get_metrics()
        plotters[f"{model}"] = temp_plotter


### Performance

In [ ]:
for model_set in ["base", "cot"]:
    models_to_plot = ["human"]
    i = 0
    if model_set == "base":
        models_to_plot += base_models
    elif model_set == "cot":
        models_to_plot += cot_models
    if len(models_to_plot) > 1:
        fig, axs = plt.subplots(2, 5, figsize=(25, 8))
        for model, plotter in plotters.items():
            if model in models_to_plot:
                early_learning = np.mean([plotter.metrics["p_correct_b2"], plotter.metrics["p_correct_b3"]], axis=0)
                late_learning = np.mean([plotter.metrics["p_correct_b6"], plotter.metrics["p_correct_b7"]], axis=0)
                learning_curve_means = np.array([np.mean(plotter.metrics[f"p_correct_b{i}"]) for i in range(1, 9)])
                learning_curve_sems = np.array([np.std(plotter.metrics[f"p_correct_b{i}"])/np.sqrt(plotter.n_subjs) for i in range(1, 9)]) 
                axs[0,0].bar(i, np.mean(plotter.metrics["p_correct_b1"]), 
                            yerr=np.std(plotter.metrics["p_correct_b1"])/np.sqrt(plotter.n_subjs), 
                            label=model_names_dict[model], color=plotter.main_color)
                axs[0,1].bar(i, np.mean(early_learning), 
                            yerr=np.std(early_learning)/np.sqrt(plotter.n_subjs), 
                            label=model_names_dict[model], color=plotter.main_color)
                axs[0,2].bar(i, np.mean(late_learning), 
                            yerr=np.std(late_learning)/np.sqrt(plotter.n_subjs), 
                            label=model_names_dict[model], color=plotter.main_color)
                axs[0,3].bar(i, np.mean(plotter.metrics["p_correct_b8_base"]), 
                            yerr=np.std(plotter.metrics["p_correct_b8_base"])/np.sqrt(plotter.n_subjs), 
                            label=model_names_dict[model], color=plotter.main_color)
                axs[0,4].bar(i, np.mean(plotter.metrics["p_correct_b8_infer"]), 
                            yerr=np.std(plotter.metrics["p_correct_b8_infer"])/np.sqrt(plotter.n_subjs), 
                            label=model_names_dict[model], color=plotter.main_color)
                axs[1,0].fill_between(np.arange(1, 9), 
                                    learning_curve_means-learning_curve_sems, 
                                    learning_curve_means+learning_curve_sems, 
                                    color=plotter.main_color, alpha=0.3)
                axs[1,0].plot(np.arange(1, 9), learning_curve_means, 
                            mec=plotter.main_color, mfc='white', markersize=6, marker=plotter.marker,
                            color=plotter.main_color, label=model_names_dict[model])
                for j, metric in enumerate([early_learning, late_learning, 
                                        plotter.metrics["p_correct_b8_base"], plotter.metrics["p_correct_b8_infer"]]):
                    y = np.sort(metric)
                    x = np.arange(1, len(y)+1) / len(y)
                    axs[1, 1+j].plot(x, y, color=plotter.main_color, label=model_names_dict[model], 
                            mec=plotter.main_color, mfc='white', markersize=6, marker=plotter.marker)
                    # Highlight example participants on sorted CDF plots
                    if highlight_examples and model in example_participants:
                        sorted_indices = np.argsort(metric)
                        for subj in example_participants[model]:
                            sorted_pos = np.where(sorted_indices == subj)[0]
                            axs[1, 1+j].plot(x[sorted_pos], y[sorted_pos],
                                    marker=plotter.marker, markersize=12,
                                    mec=plotter.main_color, mfc=plotter.main_color, zorder=10)
                i += 1
        for ax in axs[0]:
            ax.set_ylim(0.0, 1.05)
            ax.set_xticks([])
        for i, ax in enumerate(axs[1]):
            ax.set_ylim(0.0, 1.05)
            if i == 0:
                ax.set_xlabel("Block number", fontsize=18)
                ax.set_xticks(np.arange(1, 9), labels=["Practice"] + [b for b in range(1, 7)] + ["Test"])
            else:
                ax.set_xlabel("Normalized subject number", fontsize=18)

        axs[0, 0].set_ylabel("P(correct)", fontsize=18)
        axs[1, 0].set_ylabel("P(correct)", fontsize=18)
        axs[0, 0].set_title("Practice", fontsize=18)
        axs[1, 0].set_title("Learning curve", fontsize=18)
        for r in [0, 1]:
            axs[r, 1].set_title("Early learning (blocks 1-2)", fontsize=18)
            axs[r, 2].set_title("Late learning (blocks 5-6)", fontsize=18)
            axs[r, 3].set_title("Test (in distribution)", fontsize=18)
            axs[r, 4].set_title("Test (out of distribution)", fontsize=18)

        # Border around the bottom left subplot
        target_ax = axs[1, 0]
        rect = plt.Rectangle(
            (-0.23, -0.23), 1.28, 1.35,
            transform=target_ax.transAxes,
            linewidth=2, 
            edgecolor='k', 
            facecolor='none',
            clip_on=False, 
            zorder=1000
        )
        target_ax.add_patch(rect)

        learning_rand = np.average([1/12, 1/24], weights=[4, 6])
        out_test_rand = 1/12
        
        axs[1, 0].legend(bbox_to_anchor=(0.05, 2.2), loc='upper left', fontsize=18)

        fig.savefig(f"{plots_dir}{version_dir}{model_set}_models{version_suffix}{'_temp0' if temp == 0.0 else ''}_performance_metrics.png", dpi=300, bbox_inches="tight")

### Goal and action selection

In [ ]:
prefixes = {
    "p_2act_goals": "P(2-action goal chosen)", 
    "p_repeat_goal_1back": "P(repeat goal)", 
    "goal_entropy": "Goal entropy",
    "fav_goal_position": "Most chosen goal position", 
    "n_max_any_0_6_pattern": "N. of goal rotation patterns", 
    "n_systematic_hypothesis_testing": "N. of systematic actions\nby position"
    }

prefix_qs = {
    "p_2act_goals": "How often are 2-action goals chosen?", 
    "p_repeat_goal_1back": "How often is the same goal repeated?", 
    "goal_entropy": "How often and widely do goals vary?",
    "fav_goal_position": "Which goal position\nis chosen most often?", 
    "n_max_any_0_6_pattern": "How often is there\na goal rotation pattern?", 
    "n_systematic_hypothesis_testing": "Trying out permutations in order",
    }

random_lvls = {
    "p_2act_goals": 2/3, 
    "p_repeat_goal_1back": 1/6, 
    "goal_entropy": 132.2,
    "fav_goal_position": 2.5, 
    "n_max_any_0_6_pattern": 0.00298*720, 
    "n_systematic_hypothesis_testing": 0.07677,
    }

rows = 1
cols = len(prefixes)//rows
models_to_plot = plotters.keys()

for model_set in ["base", "cot"]:

    models_to_plot = ["human"]
    
    
    for model in plotters.keys():
        if model_set == "base" and ("_cot" not in model) and (model != "human"):
            models_to_plot.append(model)
        elif model_set == "cot" and ("_cot" in model):
            models_to_plot.append(model)

    if len(models_to_plot) > 1:

        fig, axs = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
        axs = axs.flatten()
        
        for i, prefix in enumerate(prefixes):
            min_val = np.inf
            max_val = -np.inf
            ax = axs[i]
            ax.set_title(prefix_qs[prefix], fontsize=18)  
            for model in models_to_plot:
                plotter = plotters[model]

                min_val = min(min_val, plotter.metrics[prefix].min())
                max_val = max(max_val, plotter.metrics[prefix].max())

                y = np.sort(plotter.metrics[prefix])
                x = np.arange(1, len(y)+1) / len(y)
                
                ax.plot(x, y, color=plotter.main_color, label=model_names_dict[model], 
                        mec=plotter.main_color, mfc='white', markersize=6, marker=plotter.marker) 

                # Highlight example participants on sorted CDF plots
                if highlight_examples and model in example_participants and model_set == "base":
                    sorted_indices = np.argsort(plotter.metrics[prefix])
                    for subj in example_participants[model]:
                        sorted_pos = np.where(sorted_indices == subj)[0]
                        ax.plot(x[sorted_pos], y[sorted_pos],
                                marker=plotter.marker, markersize=12,
                                mec=plotter.main_color, mfc=plotter.main_color, zorder=10)

                ax.set_ylabel(prefixes[prefix], fontsize=18)

            if i >= len(prefixes)-cols:
                ax.set_xlabel("Normalized subject number", fontsize=18)
            ax.set_ylim(min_val, max_val+0.05*(max_val - min_val))
            ax = axs[-1]   

        # Border around the bottom left subplot
        target_ax = axs[-1]
        rect = plt.Rectangle(
            (-0.25, -0.19), 1.3, 1.37,
            transform=target_ax.transAxes,
            linewidth=2, 
            edgecolor='k', 
            facecolor='none',
            clip_on=False, 
            zorder=1000
        )
        target_ax.add_patch(rect)

        axs[5].legend(fontsize=18, loc="upper left")
        fig.tight_layout()
        fig.savefig(f"{plots_dir}{version_dir}{model_set}_models{version_suffix}{'_temp0' if temp == 0.0 else ''}_summative_metrics_cdf3.png", dpi=300, bbox_inches="tight")

### Example participants

In [ ]:
n_models = len(base_models) + 1  # +1 for human
base_plotters = {model: plotters[model] for model in ["human"] + base_models}
n_examples = 2

act_seq_y_idx = [0, 6, 12, 18, 24, 30]

fig1, axs1 = plt.subplots(n_examples, n_models, figsize=(n_models*5, 4*n_examples))
fig2, axs2 = plt.subplots(n_examples, n_models, figsize=(n_models*5, 4*n_examples))

for p_idx, (model, plotter) in enumerate(base_plotters.items()):
    if model in example_participants:
        subjs = example_participants[model]
    elif model == "human":
        subjs = [12, 28, 173, 212][1:3]
        print(f"{model} subjects: {subjs}.")
    else:
        subjs = np.random.choice(plotter.task_df.subj_num.unique(), n_examples, replace=False)
        print(f"{model} subjects: {subjs.tolist()}.")
    for subj_idx, subj_num in enumerate(subjs):
        subj_plotter = h.Plotter(plotter.task_df.query("subj_num == @subj_num").reset_index(drop=True), 
            plotter.train_df.query("subj_num == @subj_num").reset_index(drop=True), 
            plotter.learn_df.query("subj_num == @subj_num").reset_index(drop=True), 
            plotter.test_df.query("subj_num == @subj_num").reset_index(drop=True), 
            plotting_mode=plotter.plotting_mode)
        subj_plotter.plot_trial_action_seq_indiv(axs1[subj_idx, p_idx], subj_idx=subj_idx, by_position=False, color_code="chosen_goal", 
            with_ylabel=p_idx == 0, 
            with_xlabel=subj_idx == n_examples-1, 
            legend_subj_idx=0 if (p_idx == n_examples//2+2) else None, bbox_to_anchor=(0.9,0.9), legend_fontsize=18)
        subj_plotter.plot_trial_goal_seq_indiv(axs2[subj_idx, p_idx], subj_idx=subj_idx, by_position=True,
                with_ylabel=p_idx==0, 
                with_xlabel=subj_idx == n_examples-1, 
                legend_subj_idx=0 if (p_idx == n_examples//2+2) else None, bbox_to_anchor=(1.05,0.9), legend_fontsize=18)

        if subj_idx == 0:
            axs1[subj_idx, p_idx].set_title(model_names_dict[model], fontsize=20)
            axs2[subj_idx, p_idx].set_title(model_names_dict[model], fontsize=20)

        if p_idx == 0:
            axs1[subj_idx, p_idx].set_xlabel("")
            yticks = axs1[subj_idx, p_idx].get_yticks() 
            ylabels = [label.get_text().replace(" ", "")+" " for label in axs1[subj_idx, p_idx].get_yticklabels()]
            yticks = [yticks[k] for k in act_seq_y_idx]
            ylabels = [ylabels[k] for k in act_seq_y_idx]
            lo_vals = yticks[0::2]
            lo_lbls = ylabels[0::2]
            hi_vals = yticks[1::2]
            hi_lbls = [r"$\cdot\cdot\cdot$" for i in range(len(hi_vals))]  # ylabels[1::2]
            # static axis ticks and labels
            axs1[subj_idx, p_idx].set_yticks(lo_vals+hi_vals, labels=lo_lbls+hi_lbls, fontsize=12)
            axs1[subj_idx, p_idx].set_ylabel("Action sequence", fontsize=18)
        
            axs2[subj_idx, p_idx].set_yticks(np.arange(0, 6), labels=np.arange(0, 6), fontsize=15)
            axs2[subj_idx, p_idx].set_ylabel("Chosen goal position", fontsize=18)
    axs1[1, p_idx].set_xlabel("Trial number\n(practice and learning)", fontsize=18)
    axs2[1, p_idx].set_xlabel("Trial number\n(practice and learning)", fontsize=18)
fig1.tight_layout()
fig2.tight_layout()

# Annotations
if temp == 1.0 and version_dir == "":
    axs1[0, 0].annotate('systematic\nhypothesis testing', xy=(50, 23), xytext=(65, 28), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05), ha='center')
    axs1[0, 2].annotate('2-action goals bias', xy=(70, 10), xytext=(85, 15), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))
    axs1[1, 0].annotate('high p(correct)\nlow p(repeat)\nhigh goal entropy', xy=(120, 24), xytext=(47, 34), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))
    axs1[1, 2].annotate('high p(correct)\nhigh p(repeat)\nlow goal entropy', xy=(70, 18), xytext=(85, 23), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))
    axs1[1, 3].annotate('low p(correct)\nhigh p(repeat)\nlow goal entropy', xy=(120, 2), xytext=(50, 12), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))

    axs2[0, 0].annotate('repeating goal cycle', xy=(99, 5.2), xytext=(47, 5.7), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))
    axs2[1, 3].annotate('1st goal preference', xy=(120, 0.2), xytext=(50, 1.2), fontsize=18, arrowprops=dict(facecolor='black', shrink=0.05))

fig1.savefig(f"{plots_dir}{version_dir}base_models{version_suffix}{'_temp0' if temp == 0.0 else ''}_example_participants_actions.png")
fig2.savefig(f"{plots_dir}{version_dir}base_models{version_suffix}{'_temp0' if temp == 0.0 else ''}_example_participants_goals.png")

## Quantitative comparisons

### Main

In [ ]:
def kl_divergence(p, q):
    """Compute the KL divergence between two probability distributions."""
    if np.sum(p) != 1:
        p = p / np.sum(p)
    if np.sum(q) != 1:
        q = q / np.sum(q)
    epsilon = 0.00001
    p = p+epsilon
    q = q+epsilon
    return np.sum(p*np.log(p/q))

def get_kl_divergence(metric_name, model1, model2):
    s1 = plotters[model1].metrics[metric_name]
    s2 = plotters[model2].metrics[metric_name]
    return kl_divergence(s1, s2)

def get_results(metric_name, model1, model2, test_type, **kwargs):
    return eval(f"h.print_{test_type}_results")(plotters[model1].metrics[metric_name].values, plotters[model2].metrics[metric_name].values, **kwargs)

prefixes = {
    "p_correct_learning": "P(correct) at learning", 
    "p_correct_b8_base": "P(correct) at testing\n(in distribution)", 
    "p_correct_b8_infer": "P(correct) at testing\n(out of distribution)",
    "p_2act_goals": "P(2-action goal chosen)", 
    "p_repeat_goal_1back": "P(repeat goal)", 
    "goal_entropy": "Goal entropy",
    "fav_goal_position": "Most chosen goal position", 
    "n_max_any_0_6_pattern": "N. of goal rotation patterns", 
    "n_systematic_hypothesis_testing": "N. of systematic actions\nby position",
    }

cat_vars = ["fav_goal", "fav_goal_position",]

print("="*50)
print("Learning (6 blocks) vs. humans")
print("="*50)
for model in plotters.keys():
    if model != "human":
        print(f"{model}:", end="\n    ")
        get_results([f"p_correct_b{i}" for i in range(2, 8)], "human", model, "energydistance")

for metric, name in prefixes.items():
    print("="*50)
    print(f"{name.replace('\n', ' ')} vs. humans")
    print("="*50)
    for model in plotters.keys():
        if model != "human":
            print(f"{model}:", end="\n    ")
            if metric in cat_vars:
                get_results(metric, model, "human", "chi2contingency")
            else:
                get_results(metric, model, "human", "kstest")
            print(f"    Average {'higher' if plotters[model].metrics[metric].mean() > plotters['human'].metrics[metric].mean() else 'lower'} than humans", end="\n    ")
            get_results(metric, model, "human", "mannwhitney")

print("="*50)
print("Number of times the 0-position goal was the favorite")
print("="*50)
for model in plotters.keys():
    print(f"{model}:", end="\n    ")
    n_0_fav = plotters[model].metrics['fav_goal_position'].value_counts().get(0, 0)
    print(f"{n_0_fav} time{'s' if n_0_fav != 1 else ''} = {np.round(n_0_fav / plotters[model].n_subjs * 100, 2)}%")

### Results table

#### Base

In [ ]:
prefixes = {
    "p_correct_learning": "P(corr.) learn",
    "p_correct_b8_base": "P(corr.) in-distr.",
    "p_correct_b8_infer": "P(corr.) out-distr.",
    "p_2act_goals": "P(2-act goal)",
    "p_repeat_goal_1back": "P(repeat goal)",
    "goal_entropy": "Goal entropy",
    "fav_goal_position": "Pref. goal pos.",
    "n_max_any_0_6_pattern": "N. goal cycles",
    "n_systematic_hypothesis_testing": "N. hyp. test.",
}

from IPython.display import display, HTML

model_set = "base"
include_human_descriptives = (model_set == "base" and temp == 1)
models = base_models if model_set == "base" else cot_models

desc_df, stats_df = h.build_comparison_tables(
    plotters, models, prefixes, cat_vars, model_names_dict,
    include_human_descriptives=include_human_descriptives
)

display(HTML("<h4>Descriptive statistics</h4>"))
display(desc_df.style.set_table_styles([{"selector": "th", "props": [("text-align", "center")]}, {"selector": "td", "props": [("text-align", "center")]}]))
display(HTML("<h4>Comparison to humans (* p&lt;.05, ** p&lt;.01, *** p&lt;.001)</h4>"))
display(stats_df.style.set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

#### CoT

In [ ]:
model_set = "cot"
include_human_descriptives = (model_set == "base" and temp == 1)
models = base_models if model_set == "base" else cot_models

desc_df, stats_df = h.build_comparison_tables(
    plotters, models, prefixes, cat_vars, model_names_dict,
    include_human_descriptives=include_human_descriptives
)

display(HTML("<h4>Descriptive statistics</h4>"))
display(desc_df.style.set_table_styles([{"selector": "th", "props": [("text-align", "center")]}, {"selector": "td", "props": [("text-align", "center")]}]))
display(HTML("<h4>Comparison to humans (* p&lt;.05, ** p&lt;.01, *** p&lt;.001)</h4>"))
display(stats_df.style.set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

### Closest match

#### Base

In [ ]:
model_set = "base"
all_models_to_plot = base_models if model_set == "base" else cot_models
distance_metric = "euclidean"

prefixes_dist = {
    'p_correct_learning': 'P(corr.) learn',
    'p_correct_b8_base': 'P(corr.) in-distr.',
    'p_correct_b8_infer': 'P(corr.) out-distr.',
    'p_2act_goals': 'P(2-act goal)',
    'p_repeat_goal_1back': 'P(repeat goal)',
    'goal_entropy': 'Goal entropy',
    'fav_goal_position': 'Pref. goal pos.',
    'n_max_any_0_6_pattern': 'N. goal cycles',
    'n_systematic_hypothesis_testing': 'N. hyp. test.'
}

min_dists, min_locs = h.compute_closest_human_distances(
    plotters, all_models_to_plot, prefixes_dist,
    distance_metric=distance_metric,
)

fig, ax = h.plot_closest_human_distances(
    min_dists, all_models_to_plot,
    model_names_dict, model_colors_dict, model_markers_dict, 
    distance_metric=distance_metric
)
ax.hlines(np.mean(min_dists['human']), -0.4, len(all_models_to_plot)+0.4, colors="k", linestyles="dashed", alpha=1)
fig.savefig(f"{plots_dir}{version_dir}{model_set}_models{version_suffix}{'_temp0' if temp == 0.0 else ''}_closest_human_distance.png", dpi=300, bbox_inches="tight")
plt.show()

h.print_closest_human_stats(min_dists, all_models_to_plot, model_names_dict)